# OpenPlaque — 5-Fold CV Boundary Parameter Selection

Clean Colab workflow using OpenPlaque code directly from GitHub.

**Important optimization:** nnU-Net prediction is run **once per sample case** and cached. Boundary-refinement parameters are evaluated downstream on the cached predicted masks.

Expected labeled sample-data pattern:

```text
Sample_Dataset/
  P02_LAD_axial_0000.nii.gz   # CT input image
  P02_LAD_axial.nii.gz        # expert label mask, values 0/1/2
```

or nnU-Net raw layout:

```text
Dataset001_CCTA_DHM/
  imagesTr/*_0000.nii.gz
  labelsTr/*.nii.gz
```

Outputs:

```text
/content/drive/MyDrive/OpenPlaque/CV_Boundary_Tuning/
```

Research use only. Not for clinical decision-making.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone or update OpenPlaque from GitHub.
# This notebook expects cv_boundary_tuning.py to be committed to the repository.
from pathlib import Path
import os
import sys

REPO_URL = 'https://github.com/pazzani/OpenPlaque.git'
REPO_DIR = Path('/content/OpenPlaque')

if not REPO_DIR.exists():
    !git clone {REPO_URL} /content/OpenPlaque
else:
    !git -C /content/OpenPlaque pull

SRC_DIR = REPO_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print('OpenPlaque source:', SRC_DIR)

In [ ]:
# Install requirements.
!wget -q https://raw.githubusercontent.com/pazzani/OpenPlaque/main/requirements-colab.txt -O /content/requirements-colab.txt
!pip install -q -r /content/requirements-colab.txt
!pip install -q gdown pandas scipy matplotlib SimpleITK pydicom scikit-learn

# Optional editable install if the repo is package-ready.
from pathlib import Path
if (Path('/content/OpenPlaque') / 'pyproject.toml').exists() or (Path('/content/OpenPlaque') / 'setup.py').exists():
    !pip install -q -e /content/OpenPlaque

In [ ]:
# Verify the CV module is available from the GitHub checkout.
try:
    import openplaque.cv_boundary_tuning as cvbt
    print('CV tuning module:', cvbt.__file__)
    print('Default parameter combinations:', cvbt.parameter_grid_size())
except Exception as e:
    raise ImportError(
        'Could not import openplaque.cv_boundary_tuning from the GitHub checkout.\n'
        'Commit the files from this release ZIP to the OpenPlaque repository, then rerun this cell.'
    ) from e

In [ ]:
# Configure nnU-Net environment and verify GPU.
import os
from pathlib import Path

os.environ['nnUNet_raw'] = '/content/nnUNet_raw'
os.environ['nnUNet_preprocessed'] = '/content/nnUNet_preprocessed'
os.environ['nnUNet_results'] = '/content/nnUNet_results'

for d in [os.environ['nnUNet_raw'], os.environ['nnUNet_preprocessed'], os.environ['nnUNet_results']]:
    Path(d).mkdir(parents=True, exist_ok=True)

!nvidia-smi

In [ ]:
# Extract nnU-Net model weights from Drive.
from pathlib import Path
import zipfile

model_zip = Path('/content/drive/MyDrive/OpenPlaque/models/Dataset001_CCTA_DHM-20260703T233210Z-3-001.zip')
model_target = Path('/content/nnUNet_results/Dataset001_CCTA_DHM')

if model_target.exists() and list(model_target.rglob('fold_*')):
    print('Model already extracted:', model_target)
else:
    if not model_zip.exists():
        raise FileNotFoundError(f'Missing model ZIP: {model_zip}')
    with zipfile.ZipFile(model_zip) as z:
        z.extractall('/content/nnUNet_results')
    print('Extracted model weights')

!find /content/nnUNet_results -maxdepth 4 | head -80

In [ ]:
# Locate or download the labeled sample dataset.
# This is the sample dataset used by the older boundary-tuning notebook.
from pathlib import Path
import zipfile

SAMPLE_FOLDER_URL = 'https://drive.google.com/drive/folders/1i4XD-GfP-wS50m9smGR1LzBRJokKro9_?usp=sharing'

DATASET_CANDIDATES = [
    Path('/content/sample_dataset/Sample_Dataset'),
    Path('/content/drive/MyDrive/OpenPlaque/Sample_Dataset'),
    Path('/content/drive/MyDrive/OpenPlaque/Dataset001_CCTA_DHM'),
]
DATASET_ZIP_CANDIDATES = [
    Path('/content/drive/MyDrive/OpenPlaque/Sample_Dataset.zip'),
    Path('/content/drive/MyDrive/OpenPlaque/Dataset001_CCTA_DHM.zip'),
]

dataset_dir = next((p for p in DATASET_CANDIDATES if p.exists()), None)

if dataset_dir is None:
    zip_path = next((p for p in DATASET_ZIP_CANDIDATES if p.exists()), None)
    if zip_path is not None:
        print('Extracting dataset ZIP:', zip_path)
        target = Path('/content/sample_dataset')
        target.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(target)
    else:
        print('Downloading sample dataset folder...')
        !mkdir -p /content/sample_dataset
        !gdown --folder "$SAMPLE_FOLDER_URL" -O /content/sample_dataset --remaining-ok

# Re-detect after extraction/download.
DATASET_CANDIDATES = [
    Path('/content/sample_dataset/Sample_Dataset'),
    Path('/content/sample_dataset'),
    Path('/content/drive/MyDrive/OpenPlaque/Sample_Dataset'),
    Path('/content/drive/MyDrive/OpenPlaque/Dataset001_CCTA_DHM'),
]
dataset_dir = next((p for p in DATASET_CANDIDATES if p.exists()), None)

if dataset_dir is None:
    raise FileNotFoundError('Could not locate sample dataset after extraction/download.')

print('Using dataset_dir:', dataset_dir)
!find "$dataset_dir" -maxdepth 2 -type f | head -20

In [ ]:
# Collect paired image/label cases.
from openplaque.cv_boundary_tuning import collect_sample_cases

MAX_CASES_TOTAL = None   # Use None for all cases. Set e.g. 30 for a quick test.
cases = collect_sample_cases(dataset_dir, limit=MAX_CASES_TOTAL)

print('Paired cases:', len(cases))
for c in cases[:10]:
    print(f'{c.case_id}\n  image: {c.image_path}\n  label: {c.label_path}')

In [ ]:
# Explicitly display the boundary-refinement parameter grid.
# These are post-processing parameters; nnU-Net weights are fixed.
from openplaque.cv_boundary_tuning import DEFAULT_GRID, parameter_grid_dataframe, parameter_grid_size

parameter_grid = {
    'min_component_voxels': [1, 5, 10, 25, 50],
    'lumen_distance_voxels': [0, 1, 2],
    'high_hu_threshold': [None, 700, 850, 1000],
    'low_hu_threshold': [None, -100, -50],
    'erode_core': [False],
    'erosion_iterations': [1],
}

parameter_grid_df = parameter_grid_dataframe(parameter_grid)
print('Parameter combinations:', parameter_grid_size(parameter_grid))
display(parameter_grid_df.head(20))

In [ ]:
# Stage 1: Generate or reuse the nnU-Net prediction cache.
# nnU-Net is run once per case, not once per parameter combination.
from pathlib import Path
from openplaque.cv_boundary_tuning import (
    detect_available_model_folds,
    ensure_prediction_cache,
    prediction_cache_status,
)

OUTPUT_ROOT = Path('/content/drive/MyDrive/OpenPlaque/CV_Boundary_Tuning')
PRED_DIR = OUTPUT_ROOT / 'predictions'
INPUT_DIR = Path('/content/openplaque_cv_input')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

OVERWRITE_PREDICTIONS = False
MODEL_FOLDS = detect_available_model_folds('/content/nnUNet_results', dataset_name='Dataset001_CCTA_DHM')
print('Using model folds for prediction:', MODEL_FOLDS)

before_cache = prediction_cache_status(cases, PRED_DIR)
print('Predictions already cached:', int(before_cache['prediction_exists'].sum()), '/', len(before_cache))
display(before_cache.head(10))

ensure_prediction_cache(
    cases=cases,
    pred_dir=PRED_DIR,
    input_dir=INPUT_DIR,
    dataset_name_or_id='Dataset001_CCTA_DHM',
    configuration='3d_fullres',
    folds=MODEL_FOLDS,
    overwrite=OVERWRITE_PREDICTIONS,
)

after_cache = prediction_cache_status(cases, PRED_DIR)
print('Predictions cached after this cell:', int(after_cache['prediction_exists'].sum()), '/', len(after_cache))
display(after_cache.head(10))

!find "$PRED_DIR" -maxdepth 1 -type f | head -20

In [ ]:
# Stage 2: Evaluate all boundary parameters on cached predictions.
# This is CPU post-processing; it reuses PRED_DIR and does not rerun nnU-Net.
from openplaque.cv_boundary_tuning import evaluate_all_cases, aggregate_by_params

all_case_results = evaluate_all_cases(cases, PRED_DIR, parameter_grid=parameter_grid)
full_summary = aggregate_by_params(all_case_results)

print('All case/parameter rows:', len(all_case_results))
print('Unique parameter candidates:', all_case_results['candidate_id'].nunique())
display(full_summary.head(20))

In [ ]:
# Stage 3: True 5-fold parameter-selection CV.
from openplaque.cv_boundary_tuning import make_kfold_assignments, run_5fold_parameter_cv

fold_assignments = make_kfold_assignments(cases, n_splits=5, seed=17)
heldout_results, selected_by_fold, final_params = run_5fold_parameter_cv(all_case_results, fold_assignments)

print('Held-out selected rows:', len(heldout_results))
print('Final parameters selected on all cases:')
for k, v in final_params.items():
    print(f'{k}: {v}')

display(selected_by_fold)
display(heldout_results.head(20))

In [ ]:
# Save CSV/JSON/HTML reports to Drive.
from openplaque.cv_boundary_tuning import save_cv_outputs

paths = save_cv_outputs(
    all_case_results=all_case_results,
    assignments=fold_assignments,
    heldout_results=heldout_results,
    selected_by_fold=selected_by_fold,
    final_params=final_params,
    output_dir=OUTPUT_ROOT,
)

print('Saved outputs:')
for k, p in paths.items():
    print(f'{k}: {p}')

In [ ]:
# Display held-out CV performance summary.
print('Held-out CV mean metrics using parameters selected inside each training fold')
print('-' * 80)
print('Mean score:', heldout_results['score'].mean())
print('Mean Dice:', heldout_results['dice'].mean())
print('Mean IoU:', heldout_results['iou'].mean())
print('Mean absolute TPV error fraction:', heldout_results['abs_tpv_error_fraction'].mean())

print('\nFinal parameters to apply to UCLA/new patient cases:')
for k, v in final_params.items():
    print(f'{k}: {v}')